In [40]:
# Import des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

In [41]:
# Charger le dataset
data_rent = pd.read_csv('../datasets/rent_prediction.csv')
data_rent = data_rent.dropna()
data_rent.head()

,IdentifiantMaison,Chambres,Salon,SalleDeBainInterieure,Parking,Meuble,Jardin,Superficie_m2,DistanceRoute_m,Quartier,AgeMaison,LoyerMensuel_BIF
1,2,5.0,Oui,Oui,Non,Oui,Non,234.0,47.0,Rohero,37.0,2600000
2,3,3.0,Oui,Oui,Non,Non,Oui,151.0,45.0,Kinanira,23.0,1069218
3,4,5.0,Oui,Oui,Oui,Oui,Oui,227.0,174.0,Gasekebuye,22.0,2600000
6,7,3.0,Non,Oui,Non,Oui,Oui,168.0,43.0,Kamenge,30.0,945838
8,9,3.0,Oui,Non,Non,Non,Oui,150.0,155.0,Kinama,27.0,544343


In [42]:
# Inspection des donnes
data_rent.info()

<class 'pandas.core.frame.DataFrame'>
Index: 307 entries, 1 to 507
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   IdentifiantMaison      307 non-null    int64  
 1   Chambres               307 non-null    float64
 2   Salon                  307 non-null    object 
 3   SalleDeBainInterieure  307 non-null    object 
 4   Parking                307 non-null    object 
 5   Meuble                 307 non-null    object 
 6   Jardin                 307 non-null    object 
 7   Superficie_m2          307 non-null    float64
 8   DistanceRoute_m        307 non-null    float64
 9   Quartier               307 non-null    object 
 10  AgeMaison              307 non-null    float64
 11  LoyerMensuel_BIF       307 non-null    int64  
dtypes: float64(4), int64(2), object(6)
memory usage: 31.2+ KB


In [43]:
data_rent.isnull().sum()

IdentifiantMaison        0
Chambres                 0
Salon                    0
SalleDeBainInterieure    0
Parking                  0
Meuble                   0
Jardin                   0
Superficie_m2            0
DistanceRoute_m          0
Quartier                 0
AgeMaison                0
LoyerMensuel_BIF         0
dtype: int64

In [44]:
# Nettoyage des donnees

# Supprimer l'identifiant IdentifiantMaison car il n'apporte aucune information prédictive.
data_rent = data_rent.drop(
    columns=["IdentifiantMaison"]
)

In [45]:
# Traiter les valeurs manquantes
# Variables numériques → moyenne

numeric_columns = data_rent.select_dtypes(include='number')

for col in numeric_columns:
    data_rent[col] = data_rent[col].fillna(
        data_rent[col].mean()
    )
numeric_columns.columns

Index(['Chambres', 'Superficie_m2', 'DistanceRoute_m', 'AgeMaison',
       'LoyerMensuel_BIF'],
      dtype='object')

In [46]:
# Variables catégorielles → mode
categorical_columns = data_rent.select_dtypes(include='object')


for col in categorical_columns:
    data_rent[col] = data_rent[col].fillna(
        data_rent[col].mode()[0]
    )
categorical_columns.columns

Index(['Salon', 'SalleDeBainInterieure', 'Parking', 'Meuble', 'Jardin',
       'Quartier'],
      dtype='object')

In [47]:
# Verification
data_rent.isnull().sum()

Chambres                 0
Salon                    0
SalleDeBainInterieure    0
Parking                  0
Meuble                   0
Jardin                   0
Superficie_m2            0
DistanceRoute_m          0
Quartier                 0
AgeMaison                0
LoyerMensuel_BIF         0
dtype: int64

In [48]:
# Encoder les variables catégorielles et numeriques

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_columns.columns[:4]
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_columns.columns
        )
    ]
)

In [49]:
# Sélectionner les features

In [50]:
# Créer X et y
X = data_rent.drop(
    columns=["LoyerMensuel_BIF"]
)
y = data_rent["LoyerMensuel_BIF"]

In [51]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [52]:
# Entraîner un modèle
model = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),

        (
            "regressor",
            LinearRegression()
        )
    ]
)
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  Index(['Chambres', 'Superficie_m2', 'DistanceRoute_m', 'AgeMaison'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Salon', 'SalleDeBainInterieure', 'Parking', 'Meuble', 'Jardin',
       'Quartier'],
      dtype='object'))])),
                ('regressor', LinearRegression())])

In [54]:
# Prédire
y_pred = model.predict(X_test)
y_pred

array([1789687.07767392, 1582602.89744388,  356895.44110357,
        886194.54279447, 2071008.41776326, 1612147.36230958,
       2203138.40401619, 1858854.81925742, 1364029.76660363,
         28630.50236071, 2383535.72070741,  541664.83954031,
        698080.18381407,  813714.32435568, 1547738.9139502 ,
       2341817.84853475, 1053542.53258024, 1828001.9466404 ,
       1579326.96261102, 1948018.385657  ,  685756.23148067,
        770411.73128605, 2303460.92318739,  748921.16117948,
       1411690.73652101,  578082.96040975, 1033778.97238939,
        869675.71802012, 1653089.80911524,  852618.65846499,
        557870.07301497,  983432.27783036, 2298713.17678477,
       1956496.07590502,  799376.36881056, 2230959.22233314,
        692869.80912004, 2230995.21522253,  459952.77189912,
       1071336.07001066,  447529.43897946, 1088363.7277235 ,
       1326089.93754231, 1028012.07447697,  366434.80807423,
        963797.37052455, 1393576.08197301, 1814761.30689426,
        938630.71196038,

In [55]:
# Évaluer — MAE
mae = mean_absolute_error(
    y_test,
    y_pred
)

mae

186876.87358326482

In [56]:
# Évaluer — MSE
mse = mean_squared_error(
    y_test,
    y_pred
)

mse

51606167186.94648

In [57]:
# Évaluer — RMSE
rmse = np.sqrt(mse)

rmse

227169.9081897655

In [58]:
# Évaluer — R²
r2 = r2_score(
    y_test,
    y_pred
)

r2

0.9003668296354478

In [59]:
# Interpréter


In [60]:
# Sauvegarder le modèle
joblib.dump(
    model,
    "../models/linear_regression_model.pkl"
)
print("Modèle sauvegardé avec succès")

Modèle sauvegardé avec succès


In [ ]:
# Tester le modèle sauvegardé

# loaded_model = joblib.load(
#     "../models/linear_regression_model.pkl"
# )
# loaded_model.predict(
#     X_test.head()
# )